# 05 — Error-aware refinement and donor-held-out validation

This notebook is a read-only interface for the detached [refinement workflow](../docs/REFINEMENT_WORKFLOW.md). It consumes an immutable validation `CandidateEvaluationManifest`, so a frozen but unpromoted model can be reviewed without a production run or promotion manifest. It never overwrites assignments or opens the locked-test split. It builds a preregistered broad-type probability sample for weighted error estimates and a separate enriched challenge audit for finding false-positive and false-negative mechanisms. It is intentionally broad-only: it does not create the mandatory all-cell `level='specific', parent=None` review/report or per-parent subtype reviews, so its results alone cannot satisfy promotion. A reviewed cell becomes reference evidence; a final label changes only after an upstream correction is validated, frozen, promoted, and rerun.

The notebook is viewer-agnostic. Join the exported keys to OME-Zarr/OME-TIFF tiles and masks in napari, Mantis, cytomapper, or another image viewer; QuPath is optional.


In [ ]:
import os
from pathlib import Path
import sys

import duckdb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

refinement_roots = (Path.cwd(), Path.cwd().parent, Path.cwd() / 'Phenocycler_Analysis')
REFINEMENT_ROOT = next((root for root in refinement_roots if (root / 'refinement.py').exists()), None)
if REFINEMENT_ROOT is None:
    raise FileNotFoundError('cannot locate the detached refinement.py helper')
if str(REFINEMENT_ROOT) not in sys.path:
    sys.path.insert(0, str(REFINEMENT_ROOT))

from phenocycler.artifacts import RunManifest, validate_dataset_snapshot_current
from phenocycler.candidate_evaluation import load_candidate_evaluation_manifest
from phenocycler.config import load_config
from phenocycler.hierarchical_typing import TypingRegistry
from phenocycler.marker_registry import load_registry
from phenocycler.refinement_contracts import (
    ReviewLabelProvenance, blinded_review_sample_from_dataset,
    build_review_sampling_frame, review_sample_frame_fingerprint,
    validate_review_labels, validate_review_sampling_against_assignments,
)
from phenocycler.typing_model_bundle import SCORE_SEMANTICS, load_typing_model_bundle
from refinement import (
    challenge_flags,
    coverage_risk_curve,
    grouped_bootstrap_coverage_risk,
    grouped_bootstrap_metrics,
    one_vs_rest_reliability_table,
    per_class_metrics,
)

CONFIG_PATH = None
MODEL_BUNDLE_PATH = None
SOURCE_RUN_MANIFEST_PATH = None
CANDIDATE_EVALUATION_MANIFEST_PATH = None
REVIEW_BLINDING_KEY_FILE = os.environ.get('PHENOCYCLER_REVIEW_BLINDING_KEY_FILE')
if any(path is None for path in (CONFIG_PATH, MODEL_BUNDLE_PATH, SOURCE_RUN_MANIFEST_PATH, CANDIDATE_EVALUATION_MANIFEST_PATH)):
    raise ValueError('select the config, unpromoted candidate bundle, exact source RunManifest, and validation CandidateEvaluationManifest before querying assignments')
cfg = load_config(CONFIG_PATH)
marker_registry = load_registry(cfg.marker_registry)
typing_registry = TypingRegistry.from_marker_registry(marker_registry, typing_rules=cfg.typing_rules)
model_bundle = load_typing_model_bundle(
    MODEL_BUNDLE_PATH, marker_registry=marker_registry, typing_registry=typing_registry,
)
source_run_manifest = RunManifest.read_json(SOURCE_RUN_MANIFEST_PATH)
candidate_evaluation = load_candidate_evaluation_manifest(
    CANDIDATE_EVALUATION_MANIFEST_PATH, model_bundle=model_bundle,
    marker_registry=marker_registry, typing_registry=typing_registry,
    source_run_manifest=source_run_manifest, allow_locked_test=False,
)
if candidate_evaluation.split_name != 'validation':
    raise ValueError('notebook 05 opens only the donor-disjoint validation candidate artifact')
ingest_references = [reference for reference in source_run_manifest.stages if reference.stage == 'ingest']
if len(ingest_references) != 1:
    raise ValueError('source run must contain exactly one ingest stage reference')
ingest_manifest = ingest_references[0].validate_current(
    validate_stage=True, validation_mode='content',
)
if ingest_manifest.stage != 'ingest':
    raise ValueError('source ingest reference resolved to a different stage')
if ingest_manifest.expected_donors != source_run_manifest.expected_donors:
    raise ValueError('source ingest donor set differs from its run manifest')
assignment_run_id = candidate_evaluation.assignment_run_id
assignments_dir = Path(candidate_evaluation.output.root)
print(f'candidate assignment run={assignment_run_id[:12]}  donors={len(candidate_evaluation.donors)}  root={assignments_dir}')
SCORE_SEMANTICS_VALIDATED = bool(
    model_bundle.score_semantics == SCORE_SEMANTICS
    and model_bundle.feature_schema_version == 'state-safe-threshold-support-v1'
    and model_bundle.threshold_selection_provenance is not None
    and model_bundle.threshold_selection_provenance.schema_version == 3
)
if not SCORE_SEMANTICS_VALIDATED:
    raise RuntimeError('configured typing bundle does not satisfy notebook 05 probability semantics')
review_donors = tuple(candidate_evaluation.donors)
if not review_donors:
    raise ValueError('the candidate evaluation has no donor-disjoint validation cells')
print(f'validated unpromoted bundle={model_bundle.content_id[:12]}  validation donors={len(review_donors)}')


## Preflight: assignment disposition and score semantics

Counts and quantiles are computed in DuckDB from projected Parquet columns; the full assignment table is not loaded into notebook memory. The plots are descriptive. In particular, `broad_best_probability` must not be used as a posterior or as a correction threshold until the feature semantics and donor-held-out calibration tests pass.


In [ ]:
assignment_glob = (assignments_dir / "donor_id=*" / "*.parquet").as_posix()
if not assignments_dir.exists():
    raise FileNotFoundError(f"candidate typing assignments are absent: {assignments_dir}")

connection = duckdb.connect()
connection.execute(f"SET threads={int(cfg.duckdb_threads)}")
connection.register('allowed_review_donors', pd.DataFrame({'donor_id': list(review_donors)}))
status_summary = connection.execute(
    """
    SELECT CAST(a.donor_id AS VARCHAR) AS donor_id,
           a.broad_assignment_status,
           a.broad_reason,
           count(*) AS cells
    FROM read_parquet(?) a
    JOIN allowed_review_donors d ON CAST(a.donor_id AS VARCHAR) = d.donor_id
    GROUP BY 1, 2, 3
    """,
    [assignment_glob],
).fetchdf()

score_summary = connection.execute(
    """
    SELECT a.broad_assignment_status, a.broad_label, count(*) AS cells,
           approx_quantile(a.broad_best_probability, 0.10) AS score_q10,
           approx_quantile(a.broad_best_probability, 0.50) AS score_median,
           approx_quantile(a.broad_best_probability, 0.90) AS score_q90,
           coalesce(sum(count(*)) FILTER (WHERE a.broad_assignment_status = 'inferred')
             OVER (), 0) AS inferred_cells
    FROM read_parquet(?) a
    JOIN allowed_review_donors d ON CAST(a.donor_id AS VARCHAR) = d.donor_id
    GROUP BY 1, 2
    ORDER BY 1, cells DESC
    """,
    [assignment_glob],
).fetchdf()
connection.unregister('allowed_review_donors')
display(score_summary)


In [ ]:
STATUS_ORDER = ("anchor", "inferred", "other", "ambiguous", "unavailable")
STATUS_COLORS = {
    "anchor": "#2a9d8f", "inferred": "#4c78a8", "other": "#8d99ae",
    "ambiguous": "#f4a261", "unavailable": "#bdbdbd",
}
donor_status = (
    status_summary.groupby(["donor_id", "broad_assignment_status"], observed=True)["cells"]
    .sum().unstack(fill_value=0).reindex(columns=STATUS_ORDER, fill_value=0)
)
donor_fraction = donor_status.div(donor_status.sum(axis=1), axis=0)
fig, ax = plt.subplots(figsize=(13, max(5, 0.32 * len(donor_fraction))), constrained_layout=True)
left = np.zeros(len(donor_fraction), dtype=float)
for status_name in STATUS_ORDER:
    values = donor_fraction[status_name].to_numpy(dtype=float)
    ax.barh(donor_fraction.index, values, left=left, color=STATUS_COLORS[status_name], label=status_name)
    left += values
ax.set_xlim(0, 1)
ax.set_xlabel("Fraction of donor cells")
ax.set_ylabel("Donor")
ax.set_title("Broad assignment disposition — descriptive, not an abundance target")
ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1.0))
ax.grid(axis="x", alpha=0.2)
plt.show()

reason_table = (
    status_summary.groupby(["broad_reason", "broad_assignment_status"], observed=True)["cells"]
    .sum().unstack(fill_value=0).reindex(columns=STATUS_ORDER, fill_value=0)
    .sort_values(by=list(STATUS_ORDER), ascending=False)
)
display(reason_table)


### Fail-closed score gate

The gate is derived from the validated frozen-but-unpromoted model and candidate-evaluation artifacts; there is no manual override. A rules-only run retains an uncalibrated registry score and therefore fails closed. A probabilistic candidate passes this notebook preflight only when its immutable bundle declares the current state-safe feature contract, temperature-scaled softmax probabilities, donor-disjoint split, complete bootstrap ensemble, and calibration-only threshold-selection provenance schema v3. That provenance binds all five preliminary assignment gates; the probability/margin/stability grid cannot start below its corresponding preliminary seed. Background-tail p-values remain audit-only inputs.


In [ ]:
inferred_cells = int(score_summary["inferred_cells"].max()) if len(score_summary) else 0
if inferred_cells == 0:
    print("No inferred broad calls exist among validation donors in this candidate; report zero probabilistic-rescue coverage explicitly.")
    print("This is not a count of all populated broad/specific display labels: anchors, Other, uncertainty states, terminal parents, and <Parent>-unclassified fallbacks can still appear in a QuPath export.")


## Probability sample for unbiased error estimates

The donor-split schema-v2 manifest preregisters validation `n_per_stratum` and the SHA-256 commitment of a secret key of at least 16 bytes. The library deterministically samples within donor × status × predicted-label strata using HMAC-SHA256 bound to this split, path-independent candidate assignment-run ID, and broad target; callers cannot choose a salt or sample size here. Exact candidate manifest/output IDs remain separately bound by review provenance. `probability_sampling_audit` retains opaque secret-keyed strata, populations, weights, selection probabilities, method, and committed salt. Only `blinded_probability_review` is reviewer-facing, and `blinded_review_sample_from_dataset()` obtains its image locations directly from the content-validated source-ingest snapshot. It contains only donor/object keys, opaque strata, and those validated locations—never sampling design or model outputs. The same split also preregisters `release_governance.promotion_policy_content_id` and `release_governance.signing_key_sha256`; both must be nonblank for production promotion, while the external signing key is never loaded by this validation notebook. This sample estimates error; it is not enriched enough to discover every rare failure mechanism.


In [ ]:
if not REVIEW_BLINDING_KEY_FILE:
    raise ValueError('set PHENOCYCLER_REVIEW_BLINDING_KEY_FILE to the protected key file committed by donor-splits schema v2')
review_blinding_key_path = Path(REVIEW_BLINDING_KEY_FILE).expanduser()
if not review_blinding_key_path.is_file():
    raise FileNotFoundError('the protected review blinding-key file is unavailable')
connection.register('allowed_review_donors', pd.DataFrame({'donor_id': list(review_donors)}))
review_sampling_assignments = connection.execute(
    """
    SELECT CAST(a.donor_id AS VARCHAR) AS donor_id,
           CAST(a.object_id AS VARCHAR) AS object_id,
           CAST(a.broad_assignment_status AS VARCHAR) AS broad_assignment_status,
           CAST(a.broad_label AS VARCHAR) AS broad_label
    FROM read_parquet(?) a
    JOIN allowed_review_donors d ON CAST(a.donor_id AS VARCHAR) = d.donor_id
    ORDER BY donor_id, object_id
    """,
    [assignment_glob],
).fetchdf()
review_blinding_key = review_blinding_key_path.read_bytes()
try:
    probability_sampling_audit = build_review_sampling_frame(
        review_sampling_assignments, splits=model_bundle.split_manifest,
        split_name='validation',
        candidate_assignment_run_id=assignment_run_id,
        level='broad', parent=None, blinding_key=review_blinding_key,
    )
    validate_review_sampling_against_assignments(
        probability_sampling_audit, review_sampling_assignments,
        splits=model_bundle.split_manifest, split_name='validation',
        candidate_assignment_run_id=assignment_run_id,
        level='broad', parent=None, blinding_key=review_blinding_key,
    )
finally:
    del review_blinding_key
probability_sampling_audit_fingerprint = review_sample_frame_fingerprint(
    probability_sampling_audit
)
assignment_columns = [item[0] for item in connection.execute(
    """SELECT a.* FROM read_parquet(?) a
       JOIN allowed_review_donors d ON CAST(a.donor_id AS VARCHAR) = d.donor_id
       LIMIT 0""", [assignment_glob],
).description]
broad_probability_columns = sorted(column for column in assignment_columns if column.startswith('broad_probability__'))
connection.unregister('allowed_review_donors')
blinded_probability_review = blinded_review_sample_from_dataset(
    probability_sampling_audit, source_run_manifest=source_run_manifest,
    splits=model_bundle.split_manifest,
)
if (
    blinded_probability_review.attrs.get('review_context_artifact_content_id')
    != ingest_manifest.output.content_sha256
):
    raise RuntimeError('review locations are not bound to the source-ingest output')
if blinded_probability_review.duplicated(['donor_id', 'object_id']).any():
    raise RuntimeError('blinded probability review contains duplicate cell keys')
print(f"probability sample: {len(probability_sampling_audit):,} cells from {probability_sampling_audit['sampling_stratum'].nunique():,} opaque strata")
print('Reviewer-facing view contains only keys, secret-keyed opaque strata, and validated image locations.')
display(blinded_probability_review.head(20))


## Challenge sample for false positives and false negatives

The challenge pool is scanned from the full validation-donor assignment dataset, then deterministically capped within donor × failure-mode strata; it is not filtered from the much smaller probability sample. Assignment-only flags remain a conservative first pass. A complete challenge set must also join marker exclusions, segmentation split/merge risk, correction magnitude, pixel-model disagreement, image borders/density, and discovery clusters. The challenge sample finds mechanisms; its observed error fraction is not a cohort error estimate. Preserve the stamped assignment run, split, bundle, candidate-manifest, and candidate-output identifiers through review. An adjudicated diagnostic supplied to notebook 06 must also record `root_cause`, explicit `evidence_sufficient` and `catastrophic` booleans, and `challenge_disposition` (`no_catastrophic_finding` or `new_development_cycle_required`).


In [ ]:
CHALLENGE_PER_DONOR_STRATUM = 40
CHALLENGE_RANK_CONTEXT = f'{assignment_run_id}|broad|challenge-audit-v1'
score_gate_sql = 'TRUE' if SCORE_SEMANTICS_VALIDATED else 'FALSE'
challenge_margin_threshold = float(model_bundle.thresholds.inferred_margin)
challenge_probability_threshold = float(model_bundle.thresholds.inferred_probability)
connection.register('allowed_review_donors', pd.DataFrame({'donor_id': list(review_donors)}))
challenge_pool = connection.execute(
    f"""
    WITH candidates AS (
      SELECT CAST(a.donor_id AS VARCHAR) AS donor_id, a.object_id, a.broad_label, a.specific_type,
             a.broad_assignment_status, a.subtype_assignment_status, a.specific_assignment_status,
             a.broad_reason, a.subtype_reason, a.broad_anchor_classes, a.broad_authoritative_markers,
             a.broad_best_candidate, a.subtype_best_candidate, a.broad_best_probability,
             a.subtype_best_probability, a.broad_margin, a.subtype_margin, a.broad_stability,
             a.subtype_stability, a.qc_analysis_eligible,
             a.candidate_assignment_run_id, a.candidate_evaluation_split_name,
             a.typing_model_bundle_content_id,
             CASE
               WHEN NOT coalesce(a.qc_analysis_eligible, FALSE) THEN 'geometry_qc_ineligible'
               WHEN coalesce(a.broad_anchor_classes, '') LIKE '%|%' OR a.broad_reason LIKE '%conflict%' THEN 'anchor_conflict'
               WHEN coalesce(a.specific_type, '') LIKE '%-unclassified' THEN 'subtype_unresolved'
               WHEN {score_gate_sql} AND a.broad_assignment_status IN ('anchor', 'inferred') AND a.broad_margin < {challenge_margin_threshold:.17g} THEN 'accepted_low_margin'
               WHEN {score_gate_sql} AND a.broad_assignment_status IN ('ambiguous', 'unavailable') AND a.broad_best_probability >= {challenge_probability_threshold:.17g} THEN 'abstained_high_probability'
               WHEN a.broad_assignment_status = 'other' AND coalesce(a.broad_best_candidate, '') <> '' THEN 'other_review'
               ELSE 'status_reason_contradiction'
             END AS challenge_prefilter_stratum
      FROM read_parquet(?) a
      JOIN allowed_review_donors d ON CAST(a.donor_id AS VARCHAR) = d.donor_id
      WHERE NOT coalesce(a.qc_analysis_eligible, FALSE)
         OR coalesce(a.broad_anchor_classes, '') LIKE '%|%'
         OR a.broad_reason LIKE '%conflict%'
         OR coalesce(a.specific_type, '') LIKE '%-unclassified'
         OR ({score_gate_sql} AND a.broad_assignment_status IN ('anchor', 'inferred') AND a.broad_margin < {challenge_margin_threshold:.17g})
         OR ({score_gate_sql} AND a.broad_assignment_status IN ('ambiguous', 'unavailable') AND a.broad_best_probability >= {challenge_probability_threshold:.17g})
         OR (a.broad_assignment_status = 'other' AND coalesce(a.broad_best_candidate, '') <> '')
         OR (a.broad_assignment_status IN ('anchor', 'inferred') AND regexp_matches(coalesce(a.broad_reason, ''), 'conflict|failed|unavailable|ineligible'))
    ), ranked AS (
      SELECT *, count(*) OVER (PARTITION BY donor_id, challenge_prefilter_stratum) AS challenge_stratum_population,
             row_number() OVER (PARTITION BY donor_id, challenge_prefilter_stratum ORDER BY sha256(CAST(object_id AS VARCHAR) || ?)) AS challenge_rank
      FROM candidates
    )
    SELECT * EXCLUDE (challenge_rank) FROM ranked
    WHERE challenge_rank <= ?
    ORDER BY donor_id, challenge_prefilter_stratum, object_id
    """,
    [assignment_glob, CHALLENGE_RANK_CONTEXT, CHALLENGE_PER_DONOR_STRATUM],
).fetchdf()
connection.unregister('allowed_review_donors')
validate_dataset_snapshot_current(ingest_manifest.output, mode='content')
connection.register('challenge_keys', challenge_pool[['donor_id', 'object_id']])
challenge_locations = connection.execute(
    """SELECT CAST(c.donor_id AS VARCHAR) AS donor_id, c.object_id, c.image,
              c.X_centroid, c.Y_centroid, c.cell_region
       FROM read_parquet(?) c JOIN challenge_keys k
         ON CAST(c.donor_id AS VARCHAR) = k.donor_id AND c.object_id = k.object_id""",
    [(Path(ingest_manifest.output.root) / 'donor_id=*' / '*.parquet').as_posix()],
).fetchdf()
connection.unregister('challenge_keys')
validate_dataset_snapshot_current(ingest_manifest.output, mode='content')
challenge_pool = challenge_pool.merge(
    challenge_locations, on=['donor_id', 'object_id'], how='left', validate='1:1'
)
if len(challenge_pool) and challenge_pool[['image', 'X_centroid', 'Y_centroid']].isna().any(axis=None):
    raise RuntimeError('challenge sample could not recover every image location')
flagged = challenge_flags(
    challenge_pool, low_margin=challenge_margin_threshold,
    high_probability=challenge_probability_threshold,
)
flagged["review_bundle"] = "challenge_sample"
challenge_selector = flagged["candidate_fp"] | flagged["candidate_fn"]
if not SCORE_SEMANTICS_VALIDATED:
    score_flag = flagged["review_strata"].str.contains("accepted_low_margin|abstained_high_probability", regex=True, na=False)
    non_score_flag = flagged["review_strata"].str.contains("status_reason_contradiction|anchor_conflict|other_with_candidate|geometry_qc_ineligible|subtype_unresolved", regex=True, na=False)
    print(f"inactive score-only challenge flags: {(score_flag & ~non_score_flag).sum():,}")
    challenge_selector &= non_score_flag
challenge_sample = flagged.loc[challenge_selector].copy()
challenge_sample['candidate_evaluation_manifest_content_id'] = candidate_evaluation.content_id
challenge_sample['candidate_assignment_output_content_id'] = candidate_evaluation.output_content_sha256
if not SCORE_SEMANTICS_VALIDATED:
    retained_score_flag = challenge_sample["review_strata"].str.contains("accepted_low_margin|abstained_high_probability", regex=True, na=False)
    challenge_sample.loc[retained_score_flag, "review_note"] = "score component is inactive; retained for an independent challenge reason"
print(f"challenge sample: {len(challenge_sample):,} candidate cells")
display(
    challenge_sample.groupby(["candidate_fp", "candidate_fn", "review_strata"], dropna=False)
    .size().rename("cells").sort_values(ascending=False).to_frame().head(30)
)
display(challenge_sample.head(30))


## Attach detached, blinded reference labels

Provide both a Parquet/CSV ledger and its schema-v5 `ReviewLabelProvenance` JSON manifest. The reviewer must have received only `blinded_probability_review`. Provenance fingerprints both that exact reviewer frame and the separate `probability_sampling_audit`, and binds them to the donor split, exact candidate manifest/output, current assignment run, typing-model content ID, broad target, and the source-ingest output used for review locations. The manifest must declare `validation`; this notebook refuses locked-test labels. Required ledger fields are `donor_id`, `object_id`, `reference_label`, `reviewer_1`, `reviewer_2`, `adjudicated`, `root_cause`, and `evidence_sufficient`. Sampling weights and predictions are rejoined only after the ledger validates and every row is adjudicated with sufficient evidence. This notebook evaluates broad labels only, so resolved references must belong to the registered broad ontology or `Other`; it cannot replace the mandatory all-cell specific review or subtype reviews required for promotion. Existing QuPath classifications and legacy gold bundles cannot satisfy this contract.


In [ ]:
REFERENCE_LABELS_PATH = None  # e.g. Path('data/refinement/<bundle_id>/validation_labels.parquet')
REFERENCE_MANIFEST_PATH = None  # matching immutable ReviewLabelProvenance JSON
reference_labels = pd.DataFrame()
reviewed = pd.DataFrame()
METRICS_READY = False
if (REFERENCE_LABELS_PATH is None) != (REFERENCE_MANIFEST_PATH is None):
    raise ValueError('reference labels and their provenance manifest must be selected together')
if REFERENCE_LABELS_PATH is not None:
    if model_bundle is None:
        raise ValueError('manifest-backed validation requires a configured probabilistic model bundle')
    review_provenance = ReviewLabelProvenance.read_json(REFERENCE_MANIFEST_PATH)
    if review_provenance.split_name != 'validation':
        raise ValueError('notebook 05 opens only donor-disjoint validation labels')
    label_path = Path(REFERENCE_LABELS_PATH)
    reference_labels = (
        pd.read_parquet(label_path) if label_path.suffix == '.parquet' else pd.read_csv(label_path)
    )
    reference_labels = validate_review_labels(
        reference_labels, provenance=review_provenance,
        splits=model_bundle.split_manifest,
        sample_frame=probability_sampling_audit,
        reviewer_frame=blinded_probability_review,
        assignment_run_id=assignment_run_id,
        typing_model_bundle_content_id=model_bundle.content_id,
        candidate_evaluation_manifest_content_id=candidate_evaluation.content_id,
        candidate_assignment_output_content_id=candidate_evaluation.output_content_sha256,
        allow_locked_test=False,
    )
    adjudicated = reference_labels["adjudicated"].fillna(False).astype(bool)
    sufficient = adjudicated & reference_labels["evidence_sufficient"].fillna(False).astype(bool)
    unresolved_reference = reference_labels["reference_label"].fillna("").astype(str).str.strip().str.lower().isin(["", "unsure", "ambiguous", "unavailable"])
    if (sufficient & unresolved_reference).any():
        raise ValueError("evidence-sufficient adjudications require a resolved biological reference label")
    review_disposition = pd.DataFrame({
        "cells": [len(probability_sampling_audit), len(reference_labels), int(adjudicated.sum()), int(sufficient.sum())],
    }, index=["sampled", "label row present", "adjudicated", "evidence sufficient"])
    display(review_disposition)
    allowed_broad_references = {rule.name for rule in typing_registry.broad_rules} | {'Other'}
    unknown_broad_references = sorted(set(reference_labels.loc[sufficient, 'reference_label'].astype(str)) - allowed_broad_references)
    if unknown_broad_references:
        raise ValueError(f'reference labels are outside the broad ontology: {unknown_broad_references}')
    METRICS_READY = bool(adjudicated.all() and sufficient.all())
    if METRICS_READY:
        connection.register('allowed_review_donors', pd.DataFrame({'donor_id': list(review_donors)}))
        connection.register(
            'probability_keys', probability_sampling_audit[['donor_id', 'object_id']]
        )
        sample_assignments = connection.execute(
            """SELECT a.* FROM read_parquet(?) a
            JOIN allowed_review_donors d
              ON CAST(a.donor_id AS VARCHAR) = d.donor_id
            JOIN probability_keys k
              ON CAST(a.donor_id AS VARCHAR) = k.donor_id
             AND CAST(a.object_id AS VARCHAR) = k.object_id""",
            [assignment_glob],
        ).fetchdf()
        connection.unregister('probability_keys')
        connection.unregister('allowed_review_donors')
        for key_column in ('donor_id', 'object_id'):
            sample_assignments[key_column] = sample_assignments[key_column].astype(str)
        reviewed = (
            probability_sampling_audit.merge(
                reference_labels, on=['donor_id', 'object_id'],
                how='inner', validate='1:1',
            ).merge(
                sample_assignments, on=['donor_id', 'object_id'],
                how='inner', validate='1:1',
            )
        )
        if len(reviewed) != len(probability_sampling_audit):
            raise RuntimeError('post-adjudication audit/assignment join changed sample coverage')
    elif not adjudicated.all():
        print("FAIL CLOSED: every probability-sample cell needs an adjudicated row before metrics are computed.")
    elif not sufficient.all():
        print("FAIL CLOSED: evidence-insufficient rows can be nonrandom; promotion metrics require complete sufficient ascertainment or a prespecified sensitivity-bound analysis.")
else:
    print("No detached reference bundle selected; metric cells remain intentionally inactive.")


## Weighted FP/FN metrics

Abstentions count as false negatives for the true named class when sensitivity is measured; they are never treated as a spurious named-class false positive. Report coverage beside conditional accuracy. Because evidence insufficiency can correlate with ambiguity or image quality, promotion metrics remain disabled unless the complete probability sample is adjudicated and evidence-sufficient. Bootstrap donors or ROIs—not cells—for confidence intervals before promotion.


In [ ]:
if reviewed.empty or not METRICS_READY:
    print("Attach a complete adjudicated probability-sample reference bundle to compute error metrics.")
else:
    metrics = per_class_metrics(
        reviewed["reference_label"],
        reviewed["broad_label"],
        weights=reviewed["sampling_weight"],
    )
    display(metrics)
    intervals = grouped_bootstrap_metrics(
        reviewed["reference_label"], reviewed["broad_label"], reviewed["donor_id"],
        labels=metrics["label"], weights=reviewed["sampling_weight"],
        replicates=1_000, confidence=0.95, seed=20260731,
    )
    display(intervals.loc[intervals["metric"].isin(["precision", "recall", "FPR", "FNR", "NPV"])])
    plot_intervals = intervals.loc[intervals["metric"].isin(["precision", "recall"])].copy()
    label_order = metrics["label"].tolist()
    x = np.arange(len(label_order), dtype=float)
    fig, ax = plt.subplots(figsize=(12, 5), constrained_layout=True)
    for offset, metric_name, color in ((-0.12, "precision", "#4c78a8"), (0.12, "recall", "#e45756")):
        selected = plot_intervals.loc[plot_intervals["metric"].eq(metric_name)].set_index("label").reindex(label_order)
        estimate = selected["estimate"].to_numpy(dtype=float)
        errors = np.vstack([np.maximum(0, estimate - selected["ci_low"].to_numpy(dtype=float)), np.maximum(0, selected["ci_high"].to_numpy(dtype=float) - estimate)])
        ax.errorbar(x + offset, estimate, yerr=errors, fmt='o', capsize=3, color=color, label=metric_name)
    ax.set_ylim(0, 1)
    ax.set_ylabel("Weighted estimate")
    ax.set_xticks(x, labels=label_order, rotation=45, ha="right")
    ax.set_title("FP control (precision) and FN control (recall), donor-bootstrap 95% CI")
    ax.legend()
    ax.grid(axis="y", alpha=0.2)
    plt.show()

    weighted_confusion = pd.crosstab(
        reviewed["reference_label"], reviewed["broad_label"],
        values=reviewed["sampling_weight"], aggfunc="sum", dropna=False,
    ).fillna(0.0)
    row_fraction = weighted_confusion.div(weighted_confusion.sum(axis=1), axis=0)
    fig, ax = plt.subplots(figsize=(max(7, 0.8 * len(row_fraction.columns)), max(6, 0.6 * len(row_fraction))), constrained_layout=True)
    image = ax.imshow(row_fraction.to_numpy(), vmin=0, vmax=1, cmap="Blues", aspect="auto")
    ax.set_xticks(np.arange(len(row_fraction.columns)), labels=row_fraction.columns, rotation=45, ha="right")
    ax.set_yticks(np.arange(len(row_fraction.index)), labels=row_fraction.index)
    ax.set_xlabel("Candidate broad label")
    ax.set_ylabel("Adjudicated reference")
    ax.set_title("Weighted row-normalized confusion matrix")
    fig.colorbar(image, ax=ax, label="Fraction of reference class")
    plt.show()

    if SCORE_SEMANTICS_VALIDATED and broad_probability_columns:
        class_to_column = {column.split('__', 1)[1]: column for column in broad_probability_columns}
        probability_frame = reviewed.loc[:, list(class_to_column.values())].rename(
            columns={column: class_name for class_name, column in class_to_column.items()}
        )
        reliability = one_vs_rest_reliability_table(
            reviewed['reference_label'], probability_frame,
            weights=reviewed['sampling_weight'],
            score_semantics=model_bundle.score_semantics, n_bins=10,
        )
        display(reliability.groupby('label', observed=True)[['brier_score', 'expected_calibration_error']].first())
        fig, ax = plt.subplots(figsize=(7, 6), constrained_layout=True)
        for class_name, class_bins in reliability.groupby('label', observed=True):
            nonempty = class_bins['bin_weight'].gt(0)
            ax.plot(class_bins.loc[nonempty, 'mean_probability'], class_bins.loc[nonempty, 'observed_positive_fraction'], marker='o', label=class_name)
        ax.plot([0, 1], [0, 1], '--', color='black', linewidth=1)
        ax.set(xlim=(0, 1), ylim=(0, 1), xlabel='Mean predicted probability', ylabel='Observed positive fraction', title='Donor-held-out reliability by broad class')
        ax.legend(loc='upper left', bbox_to_anchor=(1.01, 1.0))
        ax.grid(alpha=0.2)
        plt.show()
    else:
        print('Reliability/Brier diagnostics require a validated probabilistic bundle and all per-class probabilities.')


## Frozen coverage and selective error

Validation evaluates the one probability threshold already frozen in the candidate model bundle; it does not expose a validation-donor threshold sweep. Fitting produces only an unfrozen candidate. Calibration selects schema-v3 threshold provenance, and the separate freeze operation attaches it without refitting coefficients, temperature, or bootstrap members. Anchors and `Other` remain fixed, conflict/unavailable abstentions remain abstentions, and only cells eligible for the probabilistic inference lane are rethresholded. Total named-call coverage and probabilistic-rescue-lane acceptance are reported separately, so fixed anchor/`Other` calls cannot conceal a zero-call rescue gate.


In [ ]:
if reviewed.empty or not METRICS_READY or not SCORE_SEMANTICS_VALIDATED:
    print("Frozen coverage-risk point disabled until reference labels and validated score semantics are both present.")
else:
    required_gate_columns = {'broad_unique_winner_pass', 'broad_margin_pass', 'broad_stability_pass', 'broad_valid_evidence_pass', 'broad_threshold_eligible'}
    missing_gate_columns = sorted(required_gate_columns - set(reviewed.columns))
    if missing_gate_columns:
        raise KeyError(f'assignments are missing inference-gate diagnostics: {missing_gate_columns}')
    structural_rescue_lane = reviewed['broad_threshold_eligible'].astype(bool)
    inference_lane = (
        structural_rescue_lane
        & reviewed['broad_unique_winner_pass'].astype(bool)
        & reviewed['broad_margin_pass'].astype(bool)
        & reviewed['broad_stability_pass'].astype(bool)
        & reviewed['broad_valid_evidence_pass'].astype(bool)
    )
    frozen_threshold = model_bundle.thresholds.inferred_probability
    reproduced = reviewed['broad_label'].copy()
    frozen_pass = reviewed['broad_best_probability'].ge(frozen_threshold)
    reproduced.loc[inference_lane & frozen_pass] = reviewed.loc[inference_lane & frozen_pass, 'broad_best_candidate']
    reproduced.loc[inference_lane & ~frozen_pass] = 'Ambiguous'
    if not reproduced.equals(reviewed['broad_label']):
        raise RuntimeError('frozen inference-lane reconstruction does not reproduce candidate broad labels')
    curve = coverage_risk_curve(
        reviewed["reference_label"], reviewed["broad_label"],
        reviewed["broad_best_probability"], thresholds=[frozen_threshold],
        weights=reviewed["sampling_weight"],
        candidate_prediction=reviewed['broad_best_candidate'], threshold_eligible=inference_lane,
    )
    curve_intervals = grouped_bootstrap_coverage_risk(
        reviewed['reference_label'], reviewed['broad_label'], reviewed['broad_best_probability'],
        reviewed['donor_id'], thresholds=[frozen_threshold], weights=reviewed['sampling_weight'],
        candidate_prediction=reviewed['broad_best_candidate'], threshold_eligible=inference_lane,
        replicates=1_000, confidence=0.95, seed=20260801,
    )
    if not curve_intervals.loc[:, curve.columns].equals(curve):
        raise AssertionError('grouped bootstrap point estimate differs from the frozen coverage-risk calculation')
    frozen_row = curve_intervals.iloc[0]
    rescue_called = inference_lane & frozen_pass
    total_named_call = ~reproduced.isin(['Ambiguous', 'Unavailable', 'Other'])
    total_weight = float(reviewed['sampling_weight'].sum())
    total_named_weight = float(reviewed.loc[total_named_call, 'sampling_weight'].sum())
    rescue_eligible_weight = float(reviewed.loc[structural_rescue_lane, 'sampling_weight'].sum())
    rescue_called_weight = float(reviewed.loc[rescue_called, 'sampling_weight'].sum())
    rescue_lane_coverage = rescue_called_weight / rescue_eligible_weight if rescue_eligible_weight > 0 else np.nan
    if not np.isclose(total_named_weight, frozen_row['called_weight']) or not np.isclose(total_named_weight / total_weight, frozen_row['coverage']):
        raise AssertionError('reported total named-call coverage differs from the frozen candidate curve')
    coverage_report = pd.DataFrame([
        {
            'scope': 'total named calls', 'eligible_cells': len(reviewed),
            'called_cells': int(total_named_call.sum()), 'eligible_weight': total_weight,
            'called_weight': total_named_weight, 'weighted_coverage': float(frozen_row['coverage']),
        },
        {
            'scope': 'probabilistic rescue lane', 'eligible_cells': int(structural_rescue_lane.sum()),
            'called_cells': int(rescue_called.sum()), 'eligible_weight': rescue_eligible_weight,
            'called_weight': rescue_called_weight, 'weighted_coverage': rescue_lane_coverage,
        },
    ]).set_index('scope')
    display(curve_intervals)
    display(coverage_report)
    if int(structural_rescue_lane.sum()) == 0:
        print('NO RESCUE-ELIGIBLE VALIDATION CELLS: total coverage comes entirely from fixed hierarchical/non-rescue dispositions.')
    elif int(rescue_called.sum()) == 0:
        print('ZERO PROBABILISTIC-RESCUE CALLS: fixed anchors/Other may still make total named-call coverage positive.')
    fig, (risk_ax, coverage_ax) = plt.subplots(1, 2, figsize=(13, 5), constrained_layout=True)
    if frozen_row['called_weight'] > 0 and np.isfinite(frozen_row['selective_error']):
        xerr = [[max(0, frozen_row['coverage'] - frozen_row['coverage_ci_low'])], [max(0, frozen_row['coverage_ci_high'] - frozen_row['coverage'])]]
        yerr = [[max(0, frozen_row['selective_error'] - frozen_row['selective_error_ci_low'])], [max(0, frozen_row['selective_error_ci_high'] - frozen_row['selective_error'])]]
        risk_ax.errorbar([frozen_row['coverage']], [frozen_row['selective_error']], xerr=xerr, yerr=yerr, fmt='*', ms=14, capsize=4, color='#d62728', label='frozen candidate point')
    else:
        risk_ax.text(0.5, 0.5, 'Frozen candidate produced zero named calls', ha='center', va='center', transform=risk_ax.transAxes)
    risk_ax.set_xlabel("Weighted total named-call coverage")
    risk_ax.set_ylabel("Weighted error among called cells")
    risk_ax.set(xlim=(0, 1), ylim=(0, 1), title="Frozen donor-held-out coverage and selective error")
    if frozen_row['called_weight'] > 0:
        risk_ax.legend()
    risk_ax.grid(alpha=0.2)
    coverage_values = [float(frozen_row['coverage']), 0.0 if not np.isfinite(rescue_lane_coverage) else rescue_lane_coverage]
    bars = coverage_ax.bar(
        ['total named calls', 'rescue gate pass'], coverage_values,
        color=['#4c78a8', '#e45756'],
    )
    coverage_ax.set(ylim=(0, 1), ylabel='Weighted coverage within scope', title='Total versus probabilistic-rescue coverage')
    coverage_ax.grid(axis='y', alpha=0.2)
    coverage_annotations = [
        f"{float(frozen_row['coverage']):.1%}\n{int(total_named_call.sum()):,}/{len(reviewed):,} cells",
        (f"{rescue_lane_coverage:.1%}\n{int(rescue_called.sum()):,}/{int(structural_rescue_lane.sum()):,} cells" if np.isfinite(rescue_lane_coverage) else 'N/A\n0/0 cells'),
    ]
    for bar, annotation in zip(bars, coverage_annotations, strict=True):
        coverage_ax.text(bar.get_x() + bar.get_width() / 2, min(0.98, bar.get_height() + 0.03), annotation, ha='center', va='bottom')
    plt.show()


## Ambiguous disposition and promotion checklist

An ambiguous cell is never assigned from a reviewer edit or highest ranked candidate. Classify the uncertainty, correct the earliest responsible stage, refit on development donors, validate on held-out donors, freeze, and rerun. The rerun may yield a named class, `<Parent>-unclassified`, `Other`, `Unavailable`, or remain `Ambiguous`.

Before promotion, confirm: stable object universe; audited exclusions; segmentation precision and recall; spillover false-signal reduction and positive retention; independent marker FP and FN estimates; invalid evidence cannot infer a type; donor-held-out broad, per-parent subtype, and all-cell specific metrics; coverage beside accuracy; completed probability review; any SOP-required challenge audit retained as a detached diagnostic rather than a headline gate; and a one-time locked-donor result. This broad-only notebook cannot satisfy that release checklist by itself.
